# Cuaderno U2-03. Representación computacional de sistemas y procesos

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2
**Unidad 2.** Herramientas computacionales para modelación y simulación
**Subtema del plan.** 2.3 Representación computacional de sistemas y procesos
**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad2/U2_03_representacion_computacional.ipynb)

La insignia anterior queda con la dirección del repositorio pendiente. El
docente reemplaza `msc-unisucre/msc2026-material` por la ruta real antes de publicar.

Este cuaderno recorre las Secciones 2.3 y 2.4 del libro. Representa
sistemas con arreglos, resuelve balances con álgebra lineal densa, mide qué
tanto de la última cifra de un resultado merece confianza y usa el cálculo
simbólico para obtener jacobianos y linealizaciones exactas. Reproduce los
Ejemplos 2.3, 2.4 y 2.5 con las cifras que el libro publica.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante debe ser capaz de lo siguiente.

1. Distinguir una vista de una copia de un arreglo y anticipar cuándo una modificación se propaga al arreglo original.
2. Evaluar un balance sobre una malla de posición y escenario mediante difusión, sin escribir bucles, como en el Listado 2.6.
3. Resolver un sistema de balances y estimar cuántas cifras de la solución merecen confianza con el Teorema 2.2.
4. Reconocer la cancelación catastrófica y reescribir una fórmula para evitarla, con las cifras del Ejemplo 2.4.
5. Obtener el jacobiano simbólico de un modelo, convertirlo en función evaluable y leer su estabilidad a partir de los autovalores.

## Puesta a punto

La primera celda instala lo que falte y la segunda fija la semilla del curso,
la paleta del libro y la función que compara cada resultado con el valor
publicado. Ningún resultado de este cuaderno depende de una ejecución
concreta.

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        *faltantes], check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
# Configuración común a todos los cuadernos del curso.
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
          "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110,
                     "font.size": 9, "axes.grid": True,
                     "grid.linewidth": 0.4, "grid.alpha": 0.5,
                     "axes.prop_cycle": plt.cycler(color=list(PALETA.values()))})


def contra_libro(nombre: str, calculado: float, publicado: float,
                 unidad: str = "", tol: float = 1e-3, relativa: bool = True,
                 exigir: bool = True, nota: str = "") -> None:
    """Compara un resultado del cuaderno con el valor que publica el libro.

    Detiene la ejecución si la diferencia excede la tolerancia y `exigir` es
    verdadero. Las magnitudes que dependen de la máquina, como los tiempos de
    ejecución, se informan con `exigir=False` y una nota que lo advierte.
    """
    error = abs(calculado - publicado)
    if relativa and publicado != 0.0:
        error = error / abs(publicado)
    ok = error <= tol
    print(f"{nombre:<44s} cuaderno {calculado:>13.6g}  "
          f"libro {publicado:>13.6g} {unidad:<9s} "
          f"{'coincide' if ok else 'DIFIERE '}{nota}")
    if exigir and not ok:
        raise AssertionError(
            f"{nombre}, el cuaderno da {calculado!r} y el libro publica "
            f"{publicado!r}, con error {error:.3e}")


print("NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| pandas", pd.__version__, "| SymPy", sp.__version__)

### Acceso a los datos

Los archivos viven en `03_cuadernos/datos/`. La función `ruta_datos` intenta
primero la ruta relativa del repositorio y, si el archivo no está, lo regenera
con la semilla del curso. Así el cuaderno corre igual en Colab, donde no
existe la carpeta, y en una instalación local. Nunca se usan rutas absolutas
del computador del docente.

In [ ]:
# Acceso a los datos. Se intenta la ruta relativa del repositorio y, si el
# archivo no existe, se regenera con la semilla del curso.
CALIBRACION = (
    (0.0, 1.33299), (10.0, 1.34782), (20.0, 1.36384), (30.0, 1.38115),
    (40.0, 1.39986), (50.0, 1.42009), (60.0, 1.44193), (70.0, 1.46554),
)


def _gen_calibracion(destino: Path) -> None:
    """Ocho patrones de sacarosa del Ejemplo 2.4 del libro."""
    pd.DataFrame(CALIBRACION,
                 columns=["solidos_brix", "indice_refraccion"]).to_csv(
        destino, index=False, encoding="utf-8")


GENERADORES = {
    "calibracion_refractometro.csv": _gen_calibracion,
}

CANDIDATAS = [Path("datos"), Path("..") / "datos",
              Path("03_cuadernos") / "datos", Path("..") / ".." / "datos"]


def ruta_datos(nombre: str) -> Path:
    """Devuelve la ruta del archivo de datos, generándolo si hace falta."""
    for base in CANDIDATAS:
        candidata = base / nombre
        if candidata.is_file():
            return candidata
    generada = Path("salida") / "datos_generados"
    generada.mkdir(parents=True, exist_ok=True)
    destino = generada / nombre
    if not destino.is_file():
        GENERADORES[nombre](destino)
    return destino

print("datos disponibles en", ruta_datos("calibracion_refractometro.csv").parent)

## 1. El arreglo n-dimensional

La Definición 2.6 del libro describe el arreglo como un bloque contiguo de
memoria acompañado de tres descriptores, que son la forma, el tipo y el vector
de zancadas. La Figura 2.6 muestra esa correspondencia y la distinción entre
una vista, que comparte el búfer, y una copia, que no. La regla operativa es
que el rebanado produce vistas, la indexación por arreglo o por máscara
produce copias, y ante la duda se llama al método de copia.

In [ ]:
malla = np.arange(12, dtype=float).reshape(3, 4)
print("forma", malla.shape, "| tipo", malla.dtype,
      "| zancadas", malla.strides, "bytes")
print("contiguo por filas:", malla.flags["C_CONTIGUOUS"])
print(malla)

vista = malla[1:, :2]            # rebanado, comparte el búfer
copia = malla[[1, 2], :2]        # indexación por arreglo, copia
print("\nla vista comparte memoria:", np.shares_memory(malla, vista))
print("la copia comparte memoria:", np.shares_memory(malla, copia))

In [ ]:
original = malla.copy()
vista[:] = -1.0
print("tras escribir en la vista, el original cambió")
print(malla)

malla[:] = original
copia[:] = -99.0
print("\ntras escribir en la copia, el original quedó intacto")
print(malla)
assert np.array_equal(malla, original), "la copia no debe tocar el original"
print("\ncambiar la forma tampoco copia:",
      np.shares_memory(malla, malla.reshape(4, 3)))

## 2. Vectorización y difusión

La Definición 2.7 del libro define la vectorización como la práctica de
formular un cálculo como una operación única sobre arreglos completos, y la
Definición 2.8 fija la regla de difusión, que alinea los ejes por la derecha,
completa la forma más corta con ejes de tamaño uno y estira sin copiar todo
eje cuyo tamaño valga uno. La regla falla explícitamente cuando dos tamaños
distintos son ambos mayores que uno, lo cual es una virtud, pues una
incompatibilidad de formas suele delatar un error de concepción del
modelo.

In [ ]:
casos = [((3, 1), (1, 4)), ((200, 1), (2000,)), ((5, 4), (4,)),
         ((3, 4), (2, 4))]
for a, b in casos:
    try:
        forma = np.broadcast_shapes(a, b)
        print(f"  {str(a):<10s} con {str(b):<10s} ->  {forma}")
    except ValueError as error:
        print(f"  {str(a):<10s} con {str(b):<10s} ->  error, {error}")

### Ejemplo 2.3 del libro reproducido

Un tramo de 80 km de río recibe una descarga con demanda bioquímica de oxígeno
inicial de 18 mg/L y un déficit inicial de 1.2 mg/L, con constantes de
desoxigenación y de reaireación de 0.35 y 0.62 por día. El Listado 2.6 evalúa
el déficit de la Ecuación 2.2 en 2000 posiciones para 200 escenarios de
velocidad media. El libro publica un tiempo crítico de viaje de 1.9222 d, un
déficit máximo de 5.185 mg/L y una concentración mínima de 2.315 mg/L. Para
0.30 m/s el punto crítico cae a 49.8 km de la descarga, y la condición de que
ese punto quede dentro del tramo exige una velocidad no mayor que
0.48 m/s.

In [ ]:
# Listado 2.6 del libro.
kd, ka, L0, D0 = 0.35, 0.62, 18.0, 1.2      # 1/d, 1/d, mg/L, mg/L
x = np.linspace(0.0, 80.0e3, 2000)          # m
u = np.linspace(0.15, 0.60, 200)            # m/s

t = x[None, :] / (u[:, None] * 86400.0)     # (200, 2000) días
D = (kd * L0 / (ka - kd)) * (np.exp(-kd * t) - np.exp(-ka * t)) \
    + D0 * np.exp(-ka * t)
OD = 7.5 - D                                # mg/L
critico = OD.min(axis=1)                    # un valor por escenario

print("forma de t   ", t.shape)
print("forma de OD  ", OD.shape)
print("forma de la reducción por escenario", critico.shape)
print("forma de la reducción por posición ", OD.min(axis=0).shape)
print("\nconfundir el eje produce un número plausible y equivocado, "
      "de modo que se comprueba la forma y no solo el valor")

In [ ]:
# El tiempo crítico anula la derivada de la Ecuación 2.2.
t_c = np.log((ka / kd) * (1.0 - D0 * (ka - kd) / (kd * L0))) / (ka - kd)
D_max = ((kd * L0 / (ka - kd)) * (np.exp(-kd * t_c) - np.exp(-ka * t_c))
         + D0 * np.exp(-ka * t_c))

contra_libro("tiempo crítico de viaje", t_c, 1.9222, "d", tol=5e-5)
contra_libro("déficit máximo", D_max, 5.185, "mg/L", tol=5e-4,
             relativa=False)
contra_libro("concentración mínima de OD", 7.5 - D_max, 2.315, "mg/L",
             tol=5e-4, relativa=False)
contra_libro("posición crítica con u = 0.30 m/s",
             0.30 * t_c * 86400.0 / 1000.0, 49.8, "km", tol=0.05,
             relativa=False)
contra_libro("velocidad máxima con el mínimo en el tramo",
             80.0e3 / (t_c * 86400.0), 0.48, "m/s", tol=0.005,
             relativa=False)

El libro compara además el costo del bucle anidado con
el de la expresión vectorizada, y publica 0.60 s frente a 0.0083 s, unas 73
veces menos. Esos tiempos dependen de la máquina y no deben citarse como
propiedad del método, de modo que aquí se miden de nuevo y se informan sin
exigir coincidencia. Lo que sí es estructural es la razón entre ambos y el
hecho de que las dos formas coincidan dígito a dígito.

In [ ]:
import timeit


def con_bucle() -> np.ndarray:
    """Bucle anidado sobre escenario y posición."""
    salida = np.empty((u.size, x.size))
    coef = kd * L0 / (ka - kd)
    for i in range(u.size):
        for j in range(x.size):
            ti = x[j] / (u[i] * 86400.0)
            salida[i, j] = (coef * (np.exp(-kd * ti) - np.exp(-ka * ti))
                            + D0 * np.exp(-ka * ti))
    return salida


def vectorizado() -> np.ndarray:
    """Expresión única sobre la malla completa, Listado 2.6."""
    ti = x[None, :] / (u[:, None] * 86400.0)
    return ((kd * L0 / (ka - kd)) * (np.exp(-kd * ti) - np.exp(-ka * ti))
            + D0 * np.exp(-ka * ti))


t_bucle = min(timeit.repeat(con_bucle, repeat=3, number=1))
t_vect = min(timeit.repeat(vectorizado, repeat=5, number=20)) / 20

print(f"evaluaciones {u.size * x.size}")
print(f"bucle anidado        {t_bucle:9.4f} s")
print(f"expresión vectorizada {t_vect:8.6f} s")
print(f"razón                 {t_bucle / t_vect:8.1f} veces")
print(f"\ndiferencia numérica máxima entre ambas formas "
      f"{np.abs(con_bucle() - vectorizado()).max():.3e} mg/L")
assert np.allclose(con_bucle(), vectorizado(), rtol=0.0, atol=1e-12), \
    "las dos formas deben coincidir dígito a dígito"
print("\nlos tiempos dependen de la máquina. El libro midió 0.60 s y "
      "0.0083 s, con una razón cercana a 73. En Colab la razón suele quedar "
      "entre 30 y 150, y ese orden de magnitud sí es estructural")

## 3. Un sistema de balances y su condicionamiento

El Listado 2.7 del libro mezcla concentrado, jugo simple y agua en una planta
de jugos para alcanzar a la vez una masa, un contenido de sólidos solubles y
una acidez objetivo. El libro publica 66.298 kg/h de concentrado,
790.055 kg/h de jugo simple y 143.646 kg/h de agua, con residuo nulo y número
de condición de 624.8.

In [ ]:
# Listado 2.7 del libro.
# Mezcla de concentrado, jugo simple y agua para 1000 kg/h a 13.0 Bx
# y 0.95 % de acidez.
A = np.array([[1.00, 1.00, 1.00],       # masa total
              [65.0, 11.0, 0.00],       # sólidos solubles, Bx
              [4.20, 0.85, 0.00]])      # acidez, %
b = np.array([1000.0, 13.0e3, 0.95e3])

m = np.linalg.solve(A, b)               # kg/h de cada corriente
residuo = np.abs(A @ m - b).max()
kappa = np.linalg.cond(A)

contra_libro("concentrado", m[0], 66.298, "kg/h", tol=2e-5)
contra_libro("jugo simple", m[1], 790.055, "kg/h", tol=2e-5)
contra_libro("agua", m[2], 143.646, "kg/h", tol=2e-5)
contra_libro("número de condición", kappa, 624.8, "", tol=2e-4)
print(f"\nresiduo máximo {residuo:.3e} kg/h, y la masa cierra en "
      f"{m.sum():.6f} kg/h")

La Definición 2.9 y el Teorema 2.2 dan la regla de
dedo. Si el número de condición es del orden de diez a la pe y los datos se
conocen con cu cifras significativas, la solución merece confianza en
aproximadamente cu menos pe cifras. Con datos de laboratorio de cuatro cifras
y condición cercana a seiscientos quedan algo menos de dos cifras confiables,
lo cual sirve para operar una línea de producción pero no para certificar una
etiqueta. La celda siguiente lo comprueba perturbando el vector de términos
independientes.

In [ ]:
generador = np.random.default_rng(SEMILLA)
perturbaciones = 2000
peor = 0.0
for _ in range(perturbaciones):
    db = b * (1.0e-4 * generador.standard_normal(3))     # 4 cifras exactas
    dm = np.linalg.solve(A, b + db) - m
    rel_x = np.linalg.norm(dm) / np.linalg.norm(m)
    rel_b = np.linalg.norm(db) / np.linalg.norm(b)
    peor = max(peor, rel_x / rel_b)

cifras = np.log10(np.linalg.norm(b) / np.linalg.norm(b) * 1.0e4) - np.log10(kappa)
print(f"amplificación observada peor caso {peor:8.2f}")
print(f"cota del Teorema 2.2, kappa =      {kappa:8.2f}")
assert peor <= kappa * 1.001, "la cota del Teorema 2.2 no puede excederse"
print(f"\ncon cuatro cifras en los datos quedan cerca de {cifras:.1f} cifras "
      "confiables en los caudales")

## 4. Aritmética de punto flotante

La Definición 2.10 del libro define el épsilon de máquina como la distancia
entre uno y el siguiente número representable, y la unidad de redondeo como su
mitad. En doble precisión el libro publica un épsilon de 2.220446049250313e-16
y una unidad de redondeo de 1.1102230246251565e-16. El número 0.1 no existe
como tal en la máquina, sino como el racional que el libro escribe con
denominador dos a la cincuenta y cinco, cuyo error relativo vale
5.551115123125783e-17.

In [ ]:
from fractions import Fraction

eps = float(np.finfo(np.float64).eps)
u_red = eps / 2.0
contra_libro("épsilon de máquina", eps, 2.220446049250313e-16, "", tol=0.0)
contra_libro("unidad de redondeo", u_red, 1.1102230246251565e-16, "", tol=0.0)

exacto = Fraction(0.1)
print("\n0.1 en la máquina vale", exacto)
assert exacto == Fraction(3602879701896397, 2 ** 55), \
    "el racional no coincide con el que publica el libro"
error_rel = float(abs(exacto - Fraction(1, 10)) / Fraction(1, 10))
contra_libro("error relativo de 0.1", error_rel, 5.551115123125783e-17, "",
             tol=0.0)
print(f"la unidad de redondeo lo acota, {error_rel:.4e} <= {u_red:.4e}")
print(f"\n0.1 + 0.2 == 0.3 es {0.1 + 0.2 == 0.3}, "
      f"y la diferencia vale {abs(0.1 + 0.2 - 0.3):.4e}")

La cancelación catastrófica no es un defecto del
computador sino del modo de escribir la fórmula. El libro cuantifica la raíz
de menor magnitud de la ecuación cuadrática con coeficiente lineal de cien
millones y publica menos 7.450580596923828e-9 con la fórmula habitual frente a
menos uno por diez a la menos ocho con la forma equivalente, con un error
relativo del 25.5 por ciento en la primera.

In [ ]:
def raiz_menor_ingenua(b_coef: float, c_coef: float = 1.0) -> float:
    """Fórmula habitual, resta dos cantidades muy próximas."""
    return (-b_coef + np.sqrt(b_coef ** 2 - 4.0 * c_coef)) / 2.0


def raiz_menor_estable(b_coef: float, c_coef: float = 1.0) -> float:
    """Forma equivalente que evita la resta de cantidades próximas."""
    return 2.0 * c_coef / (-b_coef - np.sqrt(b_coef ** 2 - 4.0 * c_coef))


r_ingenua = raiz_menor_ingenua(1.0e8)
r_estable = raiz_menor_estable(1.0e8)
error_pct = 100.0 * abs(r_ingenua - r_estable) / abs(r_estable)

contra_libro("raíz menor por la fórmula habitual", r_ingenua,
             -7.450580596923828e-09, "", tol=0.0)
contra_libro("raíz menor por la forma estable", r_estable, -1.0e-08, "",
             tol=1e-12)
contra_libro("error relativo de la fórmula habitual", error_pct, 25.5, "%",
             tol=5e-3)

### Problema 2-14 del libro

Calcule el error relativo de la fórmula habitual para la raíz menor con
coeficiente lineal de diez a la seis, diez a la ocho y diez a la diez, y
explique la tendencia.

In [ ]:
filas = []
for potencia in (6, 8, 10):
    bb = 10.0 ** potencia
    ing, est = raiz_menor_ingenua(bb), raiz_menor_estable(bb)
    filas.append({"b": f"1e{potencia}", "ingenua": ing, "estable": est,
                  "error_%": 100.0 * abs(ing - est) / abs(est)})
tabla = pd.DataFrame(filas)
print(tabla.to_string(index=False,
                      formatters={"ingenua": "{:.6e}".format,
                                  "estable": "{:.6e}".format,
                                  "error_%": "{:8.3f}".format}))
print("\nel error crece con el coeficiente porque la raíz cuadrada se acerca "
      "cada vez más al propio coeficiente y la resta cancela más cifras. "
      "Con diez a la diez no queda ninguna cifra correcta")

## 5. Ejemplo 2.4, condicionamiento de una curva de
calibración

La calibración de un refractómetro relaciona el índice de refracción medido a
20 grados Celsius con la concentración de sólidos solubles de una solución de
sacarosa, con ocho puntos entre 0 y 70 Brix e índices entre 1.33299 y 1.46554.
El libro publica un número de condición de 5.451e9 con la base cruda, de 57.55
con la base normalizada de la Ecuación 2.3 y de 3.120e17 al formar la matriz
de las ecuaciones normales, por encima del inverso del épsilon, que vale
4.504e15.

In [ ]:
calib = pd.read_csv(ruta_datos("calibracion_refractometro.csv"))
brix = calib["solidos_brix"].to_numpy()
n_ref = calib["indice_refraccion"].to_numpy()
print(calib.to_string(index=False))

GRADO = 5
media_n, desv_n = float(n_ref.mean()), float(n_ref.std())   # poblacional
z = (n_ref - media_n) / desv_n                              # Ecuación 2.3

V_cruda = np.vander(n_ref, GRADO + 1)
V_norm = np.vander(z, GRADO + 1)

contra_libro("condición de la base cruda", np.linalg.cond(V_cruda), 5.451e9,
             "", tol=5e-4)
contra_libro("condición de la base normalizada", np.linalg.cond(V_norm),
             57.55, "", tol=5e-4)
contra_libro("inverso del épsilon de máquina", 1.0 / eps, 4.504e15, "",
             tol=5e-4)

La condición de la matriz de las ecuaciones normales
queda por encima del inverso del épsilon, de modo que sus valores singulares
más pequeños ya son ruido de redondeo. El número que devuelve la biblioteca en
ese régimen depende de la implementación de álgebra lineal instalada, y por
eso el cuaderno lo informa sin exigir coincidencia y comprueba en cambio la
propiedad estructural, que es que la condición de las ecuaciones normales es
el cuadrado de la original.

In [ ]:
kappa_normales = np.linalg.cond(V_cruda.T @ V_cruda)
contra_libro("condición de las ecuaciones normales", kappa_normales, 3.120e17,
             "", tol=5e-4, exigir=False,
             nota="  (depende de la biblioteca de álgebra lineal)")
print(f"cuadrado de la condición original {np.linalg.cond(V_cruda) ** 2:.3e}")
assert kappa_normales > 1.0 / eps, \
    "las ecuaciones normales deben quedar por encima del inverso del épsilon"
print("\nla cota del Teorema 2.2 explica por qué nunca debe formarse esa "
      "matriz, pues duplica las cifras perdidas")

In [ ]:
c_svd, *_ = np.linalg.lstsq(V_cruda, brix, rcond=None)
c_norm, *_ = np.linalg.lstsq(V_norm, brix, rcond=None)
c_normales = np.linalg.solve(V_cruda.T @ V_cruda, V_cruda.T @ brix)

rejilla = np.linspace(n_ref.min(), n_ref.max(), 200)
y_svd = np.polyval(c_svd, rejilla)
y_norm = np.polyval(c_norm, (rejilla - media_n) / desv_n)
y_normales = np.polyval(c_normales, rejilla)

print(f"coeficientes en la base cruda, magnitud máxima "
      f"{np.abs(c_svd).max():.3e}")
print(f"coeficientes en la base normalizada, magnitud máxima "
      f"{np.abs(c_norm).max():.3e}")
contra_libro("magnitud de los coeficientes crudos", np.abs(c_svd).max(),
             3.0e5, "", tol=2e-2)
print(f"\ndescomposición en valores singulares frente a base normalizada, "
      f"diferencia máxima {np.abs(y_svd - y_norm).max():.3e} Brix")

dif_caminos = float(np.abs(y_normales - y_svd).max())
dis_coef = 100.0 * float(np.linalg.norm(c_normales - c_svd)
                         / np.linalg.norm(c_svd))
contra_libro("diferencia entre las dos curvas", dif_caminos, 0.0014, "Brix",
             tol=5e-2, exigir=False,
             nota="  (depende de la biblioteca de álgebra lineal)")
contra_libro("discrepancia relativa de los coeficientes", dis_coef, 93.0, "%",
             tol=5e-2, exigir=False,
             nota="  (depende de la biblioteca de álgebra lineal)")

La verificación del Ejemplo 2.4 evalúa el polinomio de
la base cruda en el índice de 1.39986, donde los términos individuales llegan a
5.96e5 y su suma vale 40.00, de modo que la evaluación pierde 4.2 cifras por
cancelación. El residuo máximo del ajuste es de 0.0018 Brix con grado cinco,
de 0.0263 Brix con grado tres y de 0.2307 Brix con grado dos.

In [ ]:
n_eval = 1.39986
terminos = np.vander([n_eval], GRADO + 1)[0] * c_svd
suma = float(terminos.sum())
perdidas = float(np.log10(np.abs(terminos).max() / abs(suma)))

print("términos individuales:", np.round(terminos, 2).tolist())
contra_libro("término de mayor magnitud", np.abs(terminos).max(), 5.96e5, "",
             tol=2e-3)
contra_libro("suma de los términos", suma, 40.00, "Brix", tol=0.005,
             relativa=False)
contra_libro("cifras perdidas por cancelación", perdidas, 4.2, "",
             tol=0.05, relativa=False)

print()
for grado in (2, 3, 5):
    c_g, *_ = np.linalg.lstsq(np.vander(n_ref, grado + 1), brix, rcond=None)
    residuo_g = float(np.abs(np.polyval(c_g, n_ref) - brix).max())
    contra_libro(f"residuo máximo con grado {grado}", residuo_g,
                 {2: 0.2307, 3: 0.0263, 5: 0.0018}[grado], "Brix",
                 tol=5e-5, relativa=False)
print("\nel grado tres ya alcanza un residuo diez veces menor que la "
      "resolución del instrumento, de modo que el grado cinco solo agrega "
      "inestabilidad sin ganancia de exactitud")

## 6. Suma compensada

El Teorema 2.4 del libro acota el error de la suma compensada por una cantidad
que no depende del número de términos, mientras que la suma secuencial ingenua
solo admite una cota proporcional a ese número. El Listado 2.8 suma un millón
de copias de 0.1, cuyo valor exacto es cien mil, y el libro publica errores
absolutos de 1.333e-6 para la suma secuencial, exactamente cero para la suma
compensada y 2.910e-11 para la reducción de NumPy, que emplea suma por
pares.

In [ ]:
# Listado 2.8 del libro.
def suma_kahan(v) -> float:
    """Suma compensada, arrastra el error de redondeo de cada término."""
    s = 0.0
    c = 0.0
    for x in v:
        y = x - c
        t = s + y
        c = (t - s) - y
        s = t
    return s


v = np.full(1_000_000, 0.1)
s_ingenua = 0.0
for x in v:
    s_ingenua += x
error = (abs(s_ingenua - 1e5), abs(suma_kahan(v) - 1e5),
         abs(float(v.sum()) - 1e5))

contra_libro("error de la suma secuencial", error[0], 1.333e-6, "", tol=1e-3)
contra_libro("error de la suma compensada", error[1], 0.0, "", tol=0.0,
             relativa=False)
contra_libro("error de la reducción de NumPy", error[2], 2.910e-11, "",
             tol=1e-2, exigir=False,
             nota="  (depende de la implementación de la reducción)")
print("\nla reducción de la biblioteca ya es cinco órdenes de magnitud mejor "
      "que el bucle ingenuo, de modo que reimplementar sumas a mano rara vez "
      "conviene")

## 7. Estructura del modelo en el código

El Listado 2.9 del libro separa los parámetros, agrupados en una estructura
inmutable con sus unidades documentadas, de la función pura que evalúa la
derivada. El decorador que congela la estructura impide modificar un parámetro
por descuido a mitad de una simulación, que es una de las formas más difíciles
de rastrear del error silencioso.

In [ ]:
# Listado 2.9 del libro.
from dataclasses import dataclass


@dataclass(frozen=True)
class Quimiostato:
    mu_max: float = 0.40        # 1/d
    K_s: float = 25.0           # mg/L
    Y: float = 0.42             # mg biomasa por mg sustrato
    D: float = 0.15             # 1/d
    S_in: float = 800.0         # mg/L


def derivada(t, y, p):
    """Devuelve dy/dt del modelo, con y = (X, S)."""
    X, S = y
    mu = p.mu_max * S / (p.K_s + S)
    return np.array([(mu - p.D) * X, p.D * (p.S_in - S) - mu * X / p.Y])


par = Quimiostato()
try:
    par.D = 0.55
except Exception as err:
    print("la estructura congelada rechaza la modificación ->",
          type(err).__name__)
print("para cambiar un parámetro se construye otra estructura:",
      Quimiostato(D=0.55))

## 8. Cálculo simbólico y paso del símbolo al código

La Sección 2.4 del libro delimita la utilidad del cálculo simbólico a tres
tareas auxiliares, que son obtener el jacobiano de un sistema, linealizar
alrededor de un punto de operación y construir soluciones de casos límite que
sirvan de referencia. El Listado 2.10 recorre el camino completo sobre el
quimiostato. El libro publica autovalores de menos 0.15 y menos 4.906 por día,
una constante de tiempo dominante de 6.67 d y un cociente cercano a 33, que
anticipa una rigidez moderada.

In [ ]:
# Listado 2.10 del libro.
X, S = sp.symbols("X S", nonnegative=True)
mu_max, K_s, Y, D, S_in = sp.symbols("mu_max K_s Y D S_in", positive=True)

mu = mu_max * S / (K_s + S)
f = sp.Matrix([(mu - D) * X, D * (S_in - S) - mu * X / Y])
J = sp.simplify(f.jacobian([X, S]))

par_sim = {mu_max: 0.40, K_s: 25.0, Y: 0.42, D: 0.15, S_in: 800.0}
J_num = sp.lambdify((X, S), J.subs(par_sim), "numpy")
autoval = np.linalg.eigvals(np.array(J_num(329.7, 15.0), dtype=float))

print("jacobiano simbólico")
sp.pprint(J)
print("\njacobiano evaluado en el estado estacionario")
print(np.array(J_num(329.7, 15.0), dtype=float).round(5))

In [ ]:
lento, rapido = np.sort(autoval)[::-1]
contra_libro("autovalor dominante", float(lento), -0.15, "1/d", tol=5e-4)
contra_libro("autovalor rápido", float(rapido), -4.906, "1/d", tol=5e-4)
contra_libro("constante de tiempo dominante", 1.0 / abs(float(lento)), 6.67,
             "d", tol=0.005, relativa=False)
contra_libro("cociente entre autovalores", abs(float(rapido) / float(lento)),
             33.0, "", tol=0.5, relativa=False)
print("\nambos autovalores son negativos, de modo que el estado estacionario "
      "con S* = 15.0 mg/L y X* = 329.7 mg/L es asintóticamente estable, y la "
      "constante de tiempo dominante fija cuánto debe durar una simulación "
      "para alcanzar el régimen permanente")

El puente entre los dos mundos es la conversión de la
expresión simbólica en una función que evalúa con las operaciones de NumPy, de
modo que admita arreglos como argumento y se ejecute a velocidad numérica. Sin
ese paso, sustituir valores uno a uno resulta cientos de veces más lento y la
ventaja del cálculo exacto se pierde en la ejecución. La celda siguiente
compara ambos caminos sobre unas pocas evaluaciones, dimensionadas para que el
cuaderno no se demore.

In [ ]:
expresion = J.subs(par_sim)[0, 1]
puntos = np.linspace(5.0, 40.0, 40)


def por_sustitucion():
    return [float(expresion.subs({X: 329.7, S: float(s)})) for s in puntos]


funcion = sp.lambdify((X, S), expresion, "numpy")


def por_lambdify():
    return funcion(329.7, puntos)


t_sub = min(timeit.repeat(por_sustitucion, repeat=3, number=1))
t_lam = min(timeit.repeat(por_lambdify, repeat=5, number=100)) / 100
print(f"sustitución uno a uno  {t_sub:9.5f} s para {puntos.size} puntos")
print(f"función convertida     {t_lam:9.7f} s para {puntos.size} puntos")
print(f"razón                  {t_sub / t_lam:9.1f} veces")
assert np.allclose(np.array(por_sustitucion()), por_lambdify()), \
    "las dos evaluaciones deben coincidir"
print("\nlos tiempos dependen de la máquina, la razón no")

## 9. Ejemplo 2.5, linealización simbólica de un
termistor

Un termistor de coeficiente negativo con resistencia nominal de 10 kiloohmios
a 25 grados Celsius y constante de 3950 K se conecta en un divisor con una
resistencia fija de 10 kiloohmios y alimentación de 3.30 V. El libro publica
una tensión de salida de 1.650 V en el punto de operación, una sensibilidad de
36.659 mV/K, una segunda derivada de menos 2.459e-4 V por kelvin al cuadrado y
un error de la aproximación lineal de 0.54 mV a 2 K, de 3.74 mV a 5 K, de
17.26 mV a 10 K y de 83.23 mV a 20 K.

In [ ]:
T = sp.symbols("T", positive=True)
R0, B, T0, Rs, Vcc = 10000.0, 3950.0, 298.15, 10000.0, 3.30   # ohm, K, K, ohm, V

R_T = R0 * sp.exp(B * (1 / T - 1 / T0))              # Ecuación 2.4
V_o = Vcc * Rs / (Rs + R_T)

sensibilidad = sp.diff(V_o, T)
segunda = sp.diff(V_o, T, 2)

# Ecuación 2.5, forma cerrada de la sensibilidad en el punto de operación.
cerrada = B * R0 * Rs * Vcc / (T0 ** 2 * (R0 + Rs) ** 2)

V_op = float(V_o.subs(T, T0))
s_op = float(sensibilidad.subs(T, T0))
s2_op = float(segunda.subs(T, T0))

contra_libro("tensión en el punto de operación", V_op, 1.650, "V", tol=5e-4)
contra_libro("sensibilidad", s_op * 1000.0, 36.659, "mV/K", tol=5e-5)
contra_libro("Ecuación 2.5 en forma cerrada", cerrada * 1000.0, 36.659,
             "mV/K", tol=5e-5)
contra_libro("segunda derivada", s2_op, -2.459e-4, "V/K2", tol=5e-4)
contra_libro("término cuadrático de Taylor a 10 K",
             0.5 * abs(s2_op) * 100.0 * 1000.0, 12.3, "mV", tol=0.05,
             relativa=False)

In [ ]:
V_num = sp.lambdify(T, V_o, "numpy")
publicado_mV = {2: 0.54, 5: 3.74, 10: 17.26, 20: 83.23}
publicado_K = {2: 0.01, 5: 0.10, 10: 0.47, 20: 2.27}
for dT in (2, 5, 10, 20):
    err = abs(float(V_num(T0 + dT)) - (V_op + s_op * dT))
    contra_libro(f"error lineal a {dT:2d} K", err * 1000.0,
                 publicado_mV[dT], "mV", tol=0.005, relativa=False)
    contra_libro(f"error equivalente a {dT:2d} K", err / abs(s_op),
                 publicado_K[dT], "K", tol=0.005, relativa=False)

La verificación del Ejemplo 2.5 fija el intervalo en el
que el error de linealidad no excede el equivalente a 0.5 K. El libro publica
que va desde 24.2 K por debajo del punto de operación hasta 10.3 K por encima,
esto es, de 1 a 35 grados Celsius. La asimetría es la información más valiosa
del ejercicio, pues la curva de respuesta es cóncava y la aproximación lineal
se degrada mucho más rápido hacia arriba que hacia abajo.

In [ ]:
from scipy.optimize import brentq


def error_equivalente(dT: float) -> float:
    """Error de linealidad expresado en kelvin equivalentes."""
    return abs(float(V_num(T0 + dT)) - (V_op + s_op * dT)) / abs(s_op)


objetivo = lambda dT: error_equivalente(dT) - 0.5
limite_inf = brentq(objetivo, -60.0, -0.001)
limite_sup = brentq(objetivo, 0.001, 30.0)

contra_libro("extremo inferior del intervalo", limite_inf, -24.2, "K",
             tol=0.05, relativa=False)
contra_libro("extremo superior del intervalo", limite_sup, 10.3, "K",
             tol=0.05, relativa=False)
print(f"\nintervalo útil de {25.0 + limite_inf:.0f} a {25.0 + limite_sup:.0f} "
      "grados Celsius")
print(f"asimetría, el extremo frío admite {abs(limite_inf) / limite_sup:.1f} "
      "veces más desviación que el caliente")

rejilla_T = np.linspace(T0 - 30.0, T0 + 25.0, 400)
fig, ax = plt.subplots(figsize=(13.5 / 2.54, 6.2 / 2.54), layout="constrained")
ax.plot(rejilla_T - 273.15, V_num(rejilla_T), color=PALETA["azul"], lw=1.4,
        label="respuesta del divisor")
ax.plot(rejilla_T - 273.15, V_op + s_op * (rejilla_T - T0),
        color=PALETA["rojo"], lw=1.2, ls="--", label="aproximación lineal")
ax.axvspan(25.0 + limite_inf, 25.0 + limite_sup, color=PALETA["verde"],
           alpha=0.10)
ax.set_xlabel("Temperatura (grados Celsius)")
ax.set_ylabel("Tensión de salida (V)")
ax.set_xlim(float(rejilla_T[0] - 273.15), float(rejilla_T[-1] - 273.15))
ax.legend(loc="upper right")
plt.show()

## 10. Coherencia dimensional

Un modelo puede ser dimensionalmente incorrecto y aun así producir números. El
Listado 2.11 del libro comprueba que una constante de tiempo formada con una
capacidad calorífica y un coeficiente convectivo tiene efectivamente dimensión
de tiempo, y devuelve 390 s. Lo importante no es el valor sino que la aserción
se ejecute cada vez que el módulo se importa.

In [ ]:
# Listado 2.11 del libro.
from sympy.physics.units import (Dimension, convert_to, joule, kelvin,
                                 kilogram, meter, second, watt)
from sympy.physics.units.systems.si import SI

h = 25 * watt / (meter**2 * kelvin)        # coeficiente convectivo
A_int = 0.48 * meter**2                    # área de intercambio
m_masa, cp = 1.2 * kilogram, 3900 * joule / (kilogram * kelvin)

tau = m_masa * cp / (h * A_int)            # constante de tiempo
dim = SI.get_dimension_system().get_dimensional_dependencies(
    Dimension(SI.get_dimensional_expr(tau)))
assert dim == {Dimension("time"): 1}, f"dimensión inesperada: {dim}"
tau_s = convert_to(tau, second)

print("constante de tiempo:", tau_s)
contra_libro("constante de tiempo", float(tau_s / second), 390.0, "s",
             tol=1e-9)

# Si alguien invierte el cociente, la comprobación falla de inmediato.
tau_mal = m_masa * cp * h * A_int
dim_mal = SI.get_dimension_system().get_dimensional_dependencies(
    Dimension(SI.get_dimensional_expr(tau_mal)))
print("\ncon el producto en lugar del cociente la dimensión resulta",
      dim_mal)
assert dim_mal != {Dimension("time"): 1}, "el producto no tiene dimensión de tiempo"

## 11. Ejercicios guiados

Seis celdas incompletas con su verificación inmediatamente después. El
cuaderno sigue ejecutándose aunque no se completen.

### Ejercicio 1. Problema 2-11, difusión de dos formas

Determine la forma de sumar un arreglo de 365 por 1 con uno de 24, e
interprete el resultado si el primero trae temperaturas diarias y el segundo un
perfil horario. Este es el Problema 2-11 del libro.

In [ ]:
# COMPLETE: construya la suma por difusión de las dos formas y guarde el
# resultado en `campo_ej`. No use bucles ni np.tile.
REVISAR_1 = False
diaria = np.linspace(24.0, 31.0, 365).reshape(365, 1)     # grados Celsius
perfil = np.linspace(-4.0, 6.0, 24)                       # grados Celsius
campo_ej = np.zeros((1, 1))       # <- reemplace por la suma por difusión

In [ ]:
if REVISAR_1:
    assert campo_ej.shape == (365, 24), f"la forma debe ser (365, 24) y es {campo_ej.shape}"
    assert np.isclose(campo_ej[0, 0], diaria[0, 0] + perfil[0])
    assert np.isclose(campo_ej[-1, -1], diaria[-1, 0] + perfil[-1])
    print("forma obtenida", campo_ej.shape)
    print(f"temperatura mínima {campo_ej.min():.2f} y máxima "
          f"{campo_ej.max():.2f} grados Celsius")
    print("\nel resultado es la temperatura horaria de todo el año, con el "
          "perfil diario montado sobre la media de cada día. La difusión "
          "alinea los ejes por la derecha, de modo que el eje de 24 se "
          "aparea con el de tamaño uno y se estira sin copiar")
else:
    print("ejercicio 1 pendiente, complete la celda y ponga REVISAR_1 = True")

### Ejercicio 2. Vista o copia

Escriba la función que dice si una operación de indexación devuelve una vista
o una copia, sin modificar el arreglo original.

In [ ]:
# COMPLETE: devuelva "vista" si el resultado comparte memoria con el original
# y "copia" en caso contrario.
REVISAR_2 = False


def clase_de(original: np.ndarray, resultado: np.ndarray) -> str:
    """Dice si el resultado es una vista o una copia del original."""
    return "copia"        # <- reemplace por la comprobación

In [ ]:
if REVISAR_2:
    base_ej = np.arange(20.0).reshape(4, 5)
    casos_ej = {"rebanado base_ej[1:3]": base_ej[1:3],
                "columna base_ej[:, 2]": base_ej[:, 2],
                "máscara base_ej[base_ej > 8]": base_ej[base_ej > 8],
                "lista de índices base_ej[[0, 2]]": base_ej[[0, 2]],
                "cambio de forma base_ej.reshape(5, 4)": base_ej.reshape(5, 4),
                "copia explícita base_ej[1:3].copy()": base_ej[1:3].copy()}
    esperado = ["vista", "vista", "copia", "copia", "vista", "copia"]
    for (nombre, resultado), esp in zip(casos_ej.items(), esperado):
        obtenido = clase_de(base_ej, resultado)
        print(f"  {nombre:<40s} {obtenido}")
        assert obtenido == esp, f"{nombre} debe ser {esp}"
    print("\nejercicio 2 correcto, el rebanado produce vistas y la "
          "indexación por máscara o por lista produce copias")
else:
    print("ejercicio 2 pendiente, complete la celda y ponga REVISAR_2 = True")

### Ejercicio 3. Problema 2-12, cifras confiables

Un sistema de balances tiene número de condición de 4.7e6 y coeficientes con
seis cifras. Estime las cifras confiables y decida si sirven para dimensionar
una tubería. Este es el Problema 2-12 del libro.

In [ ]:
# COMPLETE: aplique la regla de dedo del Teorema 2.2, según la cual quedan
# aproximadamente q - p cifras confiables cuando kappa vale diez a la p y los
# datos traen q cifras.
REVISAR_3 = False
kappa_ej, cifras_dato = 4.7e6, 6
cifras_confiables = 0.0       # <- reemplace por la estimación

In [ ]:
if REVISAR_3:
    print(f"exponente del número de condición {np.log10(kappa_ej):.2f}")
    print(f"cifras confiables estimadas {cifras_confiables:.2f}")
    assert abs(cifras_confiables - (6 - np.log10(4.7e6))) < 1e-9, \
        "aplique la regla q menos p"
    assert cifras_confiables < 1.0, \
        "el resultado debe quedar por debajo de una cifra significativa"
    print("\nla estimación resulta negativa, de modo que no queda ninguna "
          "cifra confiable. Con ese condicionamiento el resultado no sirve "
          "para dimensionar una tubería, porque el diámetro comercial cambia "
          "con la primera cifra. La salida es reformular el sistema, escalar "
          "las ecuaciones o medir con más resolución, no cambiar de "
          "algoritmo")
else:
    print("ejercicio 3 pendiente, complete la celda y ponga REVISAR_3 = True")

### Ejercicio 4. Problema 2-15, resta de entalpías

Un balance resta dos entalpías cercanas a 2750 kJ/kg que difieren en
0.8 kJ/kg. Estime las cifras que conserva la diferencia y proponga una forma
de evitar la cancelación. Este es el Problema 2-15 del libro.

In [ ]:
# COMPLETE: calcule cuántas cifras decimales significativas conserva la resta
# de dos entalpías conocidas con siete cifras significativas cada una.
REVISAR_4 = False
h1, h2 = 2750.4, 2749.6           # kJ/kg
cifras_resta = 0.0        # <- reemplace por la estimación

In [ ]:
if REVISAR_4:
    print(f"diferencia {h1 - h2:.4f} kJ/kg")
    print(f"cifras que conserva la resta {cifras_resta:.2f}")
    assert 3.0 < cifras_resta < 4.5, \
        "la resta debe conservar entre tres y cuatro cifras"
    # El remedio es no restar, sino integrar la diferencia directamente.
    cp_agua = 4.19        # kJ/(kg K)
    dT_equivalente = (h1 - h2) / cp_agua
    print(f"\nla misma diferencia obtenida como cp por delta de temperatura "
          f"vale {cp_agua * dT_equivalente:.4f} kJ/kg y conserva las siete "
          "cifras del dato, porque nunca se forman las dos entalpías grandes")
    print("ejercicio 4 correcto")
else:
    print("ejercicio 4 pendiente, complete la celda y ponga REVISAR_4 = True")

### Ejercicio 5. Problema 2-17, linealización de
Arrhenius

Linealice la ecuación de Arrhenius en 80 grados Celsius con energía de
activación de 85 kJ/mol y halle el intervalo con error de linealidad menor al
cinco por ciento. Este es el Problema 2-17 del libro.

In [ ]:
# COMPLETE: obtenga con SymPy la derivada de k(T) = A exp(-Ea/(R T)) en el
# punto de operación y guarde el valor de la pendiente en `pendiente_ej`.
REVISAR_5 = False
R_GAS, EA, A_PRE = 8.314, 85.0e3, 1.0e12      # J/(mol K), J/mol, 1/s
T_OP = 273.15 + 80.0                          # K
Tsym = sp.symbols("T_arr", positive=True)
k_expr = A_PRE * sp.exp(-EA / (R_GAS * Tsym))
pendiente_ej = 0.0        # <- reemplace por la derivada evaluada en T_OP

In [ ]:
if REVISAR_5:
    k_op = float(k_expr.subs(Tsym, T_OP))
    analitica = k_op * EA / (R_GAS * T_OP ** 2)
    assert np.isclose(pendiente_ej, analitica, rtol=1e-10), \
        "la derivada debe coincidir con k Ea sobre R T al cuadrado"
    k_num = sp.lambdify(Tsym, k_expr, "numpy")

    def error_rel(dT: float) -> float:
        return abs((k_op + pendiente_ej * dT) - float(k_num(T_OP + dT))) / float(
            k_num(T_OP + dT))

    inferior = brentq(lambda d: error_rel(d) - 0.05, -40.0, -0.01)
    superior = brentq(lambda d: error_rel(d) - 0.05, 0.01, 40.0)
    print(f"k en el punto de operación {k_op:.5e} 1/s")
    print(f"pendiente {pendiente_ej:.5e} 1/(s K)")
    print(f"intervalo con error menor al 5 por ciento, de {inferior:.2f} K a "
          f"{superior:.2f} K, esto es de {80 + inferior:.1f} a "
          f"{80 + superior:.1f} grados Celsius")
    assert abs(inferior) < superior, \
        "el error relativo crece más rápido hacia el extremo frío"
    print("\nel intervalo es asimétrico. La aproximación lineal se degrada "
          "antes hacia el extremo frío, porque el error se mide contra una "
          "constante de velocidad que allí es mucho menor, de modo que un "
          "mismo error absoluto pesa más en términos relativos")
    print("ejercicio 5 correcto")
else:
    print("ejercicio 5 pendiente, complete la celda y ponga REVISAR_5 = True")

### Ejercicio 6. Problema 2-22, vectorización de una
evapotranspiración

Vectorice un cálculo de evapotranspiración sobre 365 días por 50 estaciones y
mida con repeticiones calibradas la razón frente a los bucles anidados. Este
es el Problema 2-22 del libro. La malla se dimensiona para que el cuaderno no
tarde, y el estudiante puede escalarla si quiere.

In [ ]:
# COMPLETE: escriba la versión vectorizada de la evapotranspiración de
# Hargreaves simplificada, ET = 0.0023 * Ra * (Tmed + 17.8) * sqrt(Tmax - Tmin),
# evaluada sobre la malla de 365 días por 50 estaciones, sin bucles.
REVISAR_6 = False
dias = np.arange(365)
Ra = 32.0 + 4.0 * np.sin(2 * np.pi * (dias - 80) / 365.0)      # MJ/(m2 d)
Tmed = 24.0 + np.linspace(-3.0, 6.0, 50)[:, None] + 3.0 * np.sin(
    2 * np.pi * (dias - 60) / 365.0)                            # grados C
amplitud = 9.0 + np.linspace(0.0, 4.0, 50)[:, None] * np.ones(365)


def et_vectorizada():
    """Evapotranspiración sobre toda la malla, en mm/d."""
    return np.zeros((50, 365))      # <- reemplace por la expresión vectorizada

In [ ]:
def et_con_bucles():
    """La misma evapotranspiración con dos bucles anidados."""
    salida = np.empty((50, 365))
    for i in range(50):
        for j in range(365):
            salida[i, j] = (0.0023 * Ra[j] * (Tmed[i, j] + 17.8)
                            * np.sqrt(amplitud[i, j]))
    return salida


if REVISAR_6:
    assert et_vectorizada().shape == (50, 365), "la forma debe ser (50, 365)"
    assert np.allclose(et_vectorizada(), et_con_bucles()), \
        "las dos formas deben coincidir dígito a dígito"
    t_b = min(timeit.repeat(et_con_bucles, repeat=3, number=1))
    t_v = min(timeit.repeat(et_vectorizada, repeat=5, number=20)) / 20
    print(f"bucles anidados       {t_b:9.5f} s")
    print(f"expresión vectorizada {t_v:9.7f} s")
    print(f"razón                 {t_b / t_v:9.1f} veces")
    print(f"evapotranspiración media {et_vectorizada().mean():.3f} mm/d")
    print("\nlos tiempos dependen de la máquina. La razón queda del mismo "
          "orden que la de la Figura 2.8 del libro")
    print("ejercicio 6 correcto")
else:
    print("ejercicio 6 pendiente, complete la celda y ponga REVISAR_6 = True")

## 12. Problemas del capítulo

Los Problemas 2-11, 2-12, 2-14, 2-15, 2-17 y 2-22 quedaron resueltos en las
secciones anteriores. Se agregan aquí el 2-13 y el 2-16.

### Problema 2-13

Demuestre con el Teorema 2.2 que formar las ecuaciones normales duplica las
cifras perdidas frente a resolver los mínimos cuadrados directamente.

In [ ]:
kappa_V = np.linalg.cond(V_cruda)
kappa_VtV = np.linalg.cond(V_cruda.T @ V_cruda)
print(f"condición de V              {kappa_V:.4e}, esto es 10 a la "
      f"{np.log10(kappa_V):.2f}")
print(f"cuadrado de esa condición   {kappa_V ** 2:.4e}, esto es 10 a la "
      f"{2 * np.log10(kappa_V):.2f}")
print(f"condición calculada de VtV  {kappa_VtV:.4e}")
print("\nla cota del Teorema 2.2 amplifica la perturbación relativa por el "
      "número de condición. Como los valores singulares de la matriz de las "
      "ecuaciones normales son los cuadrados de los de V, su condición es el "
      "cuadrado de la de V y el exponente se duplica, de modo que se pierde "
      "el doble de cifras. Resolver por descomposición en valores singulares "
      "trabaja sobre V y conserva la mitad de esas cifras")
assert abs(np.log10(kappa_V ** 2) - 2 * np.log10(kappa_V)) < 1e-9

### Problema 2-16

Obtenga el jacobiano simbólico de dos tanques en serie con descarga por
orificio y discuta qué ocurre cuando un nivel tiende a cero.

In [ ]:
h1s, h2s = sp.symbols("h1 h2", positive=True)
k1s, k2s, As = sp.symbols("k1 k2 A", positive=True)

f_tanques = sp.Matrix([-k1s * sp.sqrt(h1s) / As,
                       (k1s * sp.sqrt(h1s) - k2s * sp.sqrt(h2s)) / As])
J_tanques = sp.simplify(f_tanques.jacobian([h1s, h2s]))
print("jacobiano de los dos tanques")
sp.pprint(J_tanques)

limite = sp.limit(J_tanques[0, 0], h1s, 0, "+")
print("\nlímite del elemento (1,1) cuando el nivel del primer tanque "
      "tiende a cero:", limite)
print("\nla derivada de la raíz cuadrada diverge en el origen, de modo que "
      "el jacobiano no está acotado y el modelo pierde la condición de "
      "Lipschitz. En la práctica el integrador reduce el paso sin control "
      "cerca del vaciado, y la solución es regularizar la descarga con una "
      "raíz suavizada o detener la integración cuando el nivel alcanza el "
      "umbral de la instrumentación")

## Cierre

### Lista de comprobación

Al cerrar el cuaderno el estudiante debe poder hacer lo siguiente sin
consultar la solución.

- Decir sin ejecutar si una indexación devuelve una vista o una copia, y qué consecuencia tiene al modificarla.
- Escribir un balance sobre una malla de posición y escenario con una sola expresión y comprobar la forma del resultado.
- Calcular el número de condición de un sistema de balances y traducirlo a cifras confiables.
- Reconocer una resta de cantidades próximas y reescribirla para evitar la cancelación.
- Obtener un jacobiano simbólico, convertirlo en función evaluable y leer la estabilidad en sus autovalores.

### Qué revisar en el libro si algo no salió

- Si la distinción entre vista y copia no salió, la Definición 2.6 y la Figura 2.6.
- Si la difusión no salió, la Definición 2.8 y la Figura 2.7.
- Si el condicionamiento no salió, la Definición 2.9, el Teorema 2.2 y el Ejemplo 2.4.
- Si la cancelación no salió, la Definición 2.10 y el Teorema 2.3.
- Si el jacobiano no salió, la Definición 2.11, el Listado 2.10 y el Ejemplo 2.5.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente automático de programación.
Todo fragmento se sometió al protocolo del Algoritmo 2.3 del libro y cada
resultado numérico se comprueba contra la cifra publicada mediante la función
`contra_libro`. La regla de la asignatura es que el ingeniero responde por el
resultado que firma, con independencia de quién haya tecleado las líneas.